# 📊 Analyse de Données de Vente – Online Retail Dataset
**Big Data Project – PySpark + HDFS**

## 1️⃣ Initialisation de la session Spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Analyse Ventes Online Retail") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("✅ Spark démarré avec succès !")
print(f"Version Spark : {spark.version}")

## 2️⃣ Chargement des données
> ⚠️ Place le fichier `Online Retail.xlsx` ou `online_retail.csv` dans le dossier `data/`

In [ ]:
# Chargement depuis le fichier local (dossier data/)
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", "\"") \
    .csv("/home/jovyan/data/online_retail.csv")

print(f"✅ Données chargées : {df.count()} lignes")
print(f"📋 Colonnes : {df.columns}")
df.printSchema()

In [ ]:
# Aperçu des premières lignes
df.show(5, truncate=False)

## 3️⃣ Nettoyage des données

In [ ]:
# Vérification des valeurs nulles
print("🔍 Valeurs nulles par colonne :")
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

In [ ]:
# Nettoyage :
# - Suppression des lignes sans CustomerID
# - Suppression des quantités négatives (retours)
# - Suppression des prix nuls ou négatifs
# - Suppression des doublons

df_clean = df \
    .filter(F.col("CustomerID").isNotNull()) \
    .filter(F.col("Quantity") > 0) \
    .filter(F.col("UnitPrice") > 0) \
    .dropDuplicates()

# Ajout d'une colonne TotalPrice
df_clean = df_clean.withColumn("TotalPrice", F.col("Quantity") * F.col("UnitPrice"))

# Conversion de la date
df_clean = df_clean.withColumn("InvoiceDate", F.to_timestamp("InvoiceDate", "yyyy-MM-dd HH:mm"))
df_clean = df_clean.withColumn("Month", F.month("InvoiceDate"))
df_clean = df_clean.withColumn("Year", F.year("InvoiceDate"))

print(f"✅ Après nettoyage : {df_clean.count()} lignes")
df_clean.show(5, truncate=False)

## 4️⃣ Analyses des ventes

In [ ]:
# ── Chiffre d'affaires total ──
ca_total = df_clean.agg(F.round(F.sum("TotalPrice"), 2).alias("CA_Total")).collect()[0][0]
print(f"💰 Chiffre d'affaires total : {ca_total} £")

In [ ]:
# ── Top 10 produits les plus vendus (en quantité) ──
print("🏆 Top 10 produits les plus vendus :")
df_clean.groupBy("StockCode", "Description") \
    .agg(F.sum("Quantity").alias("Total_Vendu")) \
    .orderBy(F.desc("Total_Vendu")) \
    .show(10, truncate=False)

In [ ]:
# ── Top 10 produits par chiffre d'affaires ──
print("💵 Top 10 produits par CA :")
df_clean.groupBy("StockCode", "Description") \
    .agg(F.round(F.sum("TotalPrice"), 2).alias("CA")) \
    .orderBy(F.desc("CA")) \
    .show(10, truncate=False)

In [ ]:
# ── Chiffre d'affaires par pays ──
print("🌍 CA par pays (Top 10) :")
df_clean.groupBy("Country") \
    .agg(F.round(F.sum("TotalPrice"), 2).alias("CA")) \
    .orderBy(F.desc("CA")) \
    .show(10, truncate=False)

In [ ]:
# ── Évolution mensuelle du CA ──
print("📅 Évolution mensuelle du CA :")
df_clean.groupBy("Year", "Month") \
    .agg(F.round(F.sum("TotalPrice"), 2).alias("CA_Mensuel")) \
    .orderBy("Year", "Month") \
    .show(20)

In [ ]:
# ── Top 10 clients par CA ──
print("👤 Top 10 clients par CA :")
df_clean.groupBy("CustomerID") \
    .agg(F.round(F.sum("TotalPrice"), 2).alias("CA_Client")) \
    .orderBy(F.desc("CA_Client")) \
    .show(10)

## 5️⃣ Indicateurs clés (KPIs)

In [ ]:
# ── Nombre de commandes uniques ──
nb_commandes = df_clean.select("InvoiceNo").distinct().count()
print(f"🧾 Nombre de commandes uniques : {nb_commandes}")

# ── Nombre de clients uniques ──
nb_clients = df_clean.select("CustomerID").distinct().count()
print(f"👥 Nombre de clients uniques : {nb_clients}")

# ── Nombre de produits uniques ──
nb_produits = df_clean.select("StockCode").distinct().count()
print(f"📦 Nombre de produits uniques : {nb_produits}")

# ── Panier moyen par commande ──
panier_moyen = df_clean.groupBy("InvoiceNo") \
    .agg(F.sum("TotalPrice").alias("Total_Commande")) \
    .agg(F.round(F.avg("Total_Commande"), 2).alias("Panier_Moyen")) \
    .collect()[0][0]
print(f"🛒 Panier moyen : {panier_moyen} £")

# ── Nombre moyen d'articles par commande ──
articles_moyen = df_clean.groupBy("InvoiceNo") \
    .agg(F.sum("Quantity").alias("NbArticles")) \
    .agg(F.round(F.avg("NbArticles"), 1).alias("Moy_Articles")) \
    .collect()[0][0]
print(f"📦 Nombre moyen d'articles par commande : {articles_moyen}")

## 6️⃣ Stockage sur HDFS

In [ ]:
# Sauvegarde des données nettoyées sur HDFS
hdfs_path = "hdfs://namenode:9000/data/online_retail_clean"

df_clean.write \
    .mode("overwrite") \
    .parquet(hdfs_path)

print(f"✅ Données sauvegardées sur HDFS : {hdfs_path}")

In [ ]:
# Vérification : relecture depuis HDFS
df_hdfs = spark.read.parquet(hdfs_path)
print(f"✅ Lecture depuis HDFS : {df_hdfs.count()} lignes")
df_hdfs.show(5)

## 7️⃣ Fin de session

In [ ]:
spark.stop()
print("✅ Session Spark arrêtée.")